# Supabase Modeling EDA

This notebook is the first modeling lab for the Polymarket BTC 5m system. It reads the cleaned Supabase views, checks whether the dataset is ready, studies prediction quality, and highlights where the model is gaining or losing edge.

Run this after Docker/collector has been running and after `migrations/002_modeling_views.sql` exists in Supabase.

## Setup

The notebook uses the same `.env` as the app. Keep the service role key only on your machine/backend; never put it in frontend code.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'db.py').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent.resolve()
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import db

pd.set_option('display.max_columns', 120)
sns.set_theme(style='whitegrid')

print('Project:', PROJECT_ROOT)
print('Supabase enabled:', db.db_enabled())

## Dataset Health

These counts tell us whether the collector is filling the warehouse. `round_results` is the key table for supervised learning because it gives us the final UP/DOWN target.

In [ ]:
TABLES = [
    'round_snapshots',
    'polymarket_quotes',
    'model_predictions',
    'simulated_bets',
    'round_results',
    'polymarket_markets',
]

counts = {table: db.count_rows(table) for table in TABLES}
pd.Series(counts, name='rows').to_frame()

## Load Modeling View

`modeling_snapshots` joins prediction-time state with the final round result. This is the table/view we should use for experiments because it avoids hand-joining raw telemetry in every notebook.

In [ ]:
rows = db._get(
    'modeling_snapshots',
    {
        'select': '*',
        'order': 'observed_at.asc',
        'limit': 10000,
    },
)
df = pd.DataFrame(rows)

if df.empty:
    raise RuntimeError('No rows returned from modeling_snapshots. Check the migration and collector.')

df['observed_at'] = pd.to_datetime(df['observed_at'], utc=True)
for col in ['seconds_to_cutoff', 'btc_price', 'baseline', 'dist_to_baseline', 'dist_to_baseline_pct', 'prob_up', 'prob_down', 'confidence', 'edge_up', 'edge_down', 'target_up']:
    if col in df:
        df[col] = pd.to_numeric(df[col], errors='coerce')

resolved = df[df['target_up'].notna()].copy()
print('modeling rows:', len(df))
print('resolved rows:', len(resolved))
display(df.tail(5))

## Baseline Prediction Quality

This section evaluates the currently logged model predictions. It does not retrain anything; it asks: when the live system said UP/DOWN, what happened?

In [ ]:
eval_df = resolved.dropna(subset=['target_up', 'prob_up']).copy()
eval_df['pred_up'] = (eval_df['prob_up'] >= 0.5).astype(int)
eval_df['correct'] = eval_df['pred_up'].eq(eval_df['target_up'].astype(int))

summary = {
    'rows': len(eval_df),
    'rounds': eval_df['round_cutoff'].nunique(),
    'accuracy': eval_df['correct'].mean(),
    'avg_prob_up': eval_df['prob_up'].mean(),
    'up_rate_actual': eval_df['target_up'].mean(),
}
pd.Series(summary).to_frame('value')

## Performance By Seconds Remaining

This tells us whether decisions are better right after the baseline is created or later in the round. This matters because your real question is: should I enter now?

In [ ]:
time_perf = eval_df.copy()
time_perf['seconds_bucket'] = pd.cut(
    time_perf['seconds_to_cutoff'],
    bins=[0, 30, 60, 90, 120, 180, 240, 300],
    include_lowest=True,
)

bucket_summary = (
    time_perf.groupby('seconds_bucket', observed=True)
    .agg(rows=('correct', 'size'), rounds=('round_cutoff', 'nunique'), accuracy=('correct', 'mean'), avg_confidence=('confidence', 'mean'))
    .reset_index()
)
display(bucket_summary)

ax = sns.barplot(data=bucket_summary, x='seconds_bucket', y='accuracy', color='#2f80ed')
ax.axhline(0.5, color='black', linewidth=1, linestyle='--')
ax.set_ylim(0, 1)
ax.set_title('Prediction accuracy by seconds remaining')
ax.tick_params(axis='x', rotation=35)
plt.show()

## Simulated Trading Performance

Here we look at signals that crossed the edge threshold. This is closer to strategy quality than raw model accuracy.

In [ ]:
bets = pd.DataFrame(db._get('simulated_bet_performance', {'select': '*', 'order': 'observed_at.asc', 'limit': 10000}))

if bets.empty:
    print('No simulated bets yet.')
else:
    bets['observed_at'] = pd.to_datetime(bets['observed_at'], utc=True)
    for col in ['entry_price', 'stake', 'model_prob', 'edge', 'pnl', 'seconds_to_cutoff', 'baseline', 'btc_price', 'actual_close']:
        if col in bets:
            bets[col] = pd.to_numeric(bets[col], errors='coerce')
    closed = bets[bets['result'].isin(['WIN', 'LOSS'])].copy()
    closed['win'] = closed['result'].eq('WIN')
    closed['cum_pnl'] = closed['pnl'].fillna(0).cumsum()
    display(closed.tail(10))
    display(pd.Series({
        'closed_bets': len(closed),
        'win_rate': closed['win'].mean() if len(closed) else np.nan,
        'total_pnl': closed['pnl'].fillna(0).sum(),
        'avg_pnl': closed['pnl'].mean() if len(closed) else np.nan,
    }).to_frame('value'))
    if len(closed):
        closed.plot(x='observed_at', y='cum_pnl', figsize=(10, 4), title='Cumulative simulated PnL')
        plt.show()

## Edge Buckets

If the model is useful for trading, higher edge buckets should usually have better win rate or PnL. If not, the probability calibration is not trustworthy yet.

In [ ]:
if 'closed' in globals() and len(closed):
    edge_df = closed.dropna(subset=['edge']).copy()
    edge_df['edge_bucket'] = pd.cut(edge_df['edge'], bins=[-1, 0, 0.03, 0.06, 0.10, 0.15, 1], include_lowest=True)
    edge_summary = (
        edge_df.groupby(['side', 'edge_bucket'], observed=True)
        .agg(bets=('id', 'size'), win_rate=('win', 'mean'), total_pnl=('pnl', 'sum'), avg_entry=('entry_price', 'mean'))
        .reset_index()
    )
    display(edge_summary)
else:
    print('Need closed simulated bets before edge bucket analysis.')

## Feature Table For The Next Model

This flattens `feature_values` plus prediction-time columns into a single matrix. Use this to inspect missing values, correlations, and candidate variables before training.

In [ ]:
feature_values = pd.json_normalize([value or {} for value in resolved['feature_values']])
base_cols = ['seconds_to_cutoff', 'btc_price', 'baseline', 'dist_to_baseline', 'dist_to_baseline_pct', 'prob_up', 'prob_down', 'confidence', 'edge_up', 'edge_down']
X = pd.concat([resolved[base_cols].reset_index(drop=True), feature_values.reset_index(drop=True)], axis=1)
X = X.apply(pd.to_numeric, errors='coerce').replace([np.inf, -np.inf], np.nan)
y = resolved['target_up'].astype(int).reset_index(drop=True)

quality = pd.DataFrame({
    'missing_pct': X.isna().mean().sort_values(ascending=False),
    'nunique': X.nunique(dropna=True),
}).sort_values(['missing_pct', 'nunique'], ascending=[False, True])

display(X.head())
display(quality.head(30))

## Simple Walk-Forward Check

This is intentionally small. The goal is to validate the experiment loop, not to claim a production-grade model. For serious tuning, collect more rounds first.

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import accuracy_score, brier_score_loss, log_loss, roc_auc_score

model_df = pd.concat([resolved[['observed_at', 'round_cutoff']].reset_index(drop=True), X.fillna(0).reset_index(drop=True)], axis=1)
model_df['target_up'] = y
model_df = model_df.sort_values('observed_at').reset_index(drop=True)

if len(model_df) < 500:
    print(f'Only {len(model_df)} resolved rows. Keep collecting before trusting training metrics.')
else:
    split = int(len(model_df) * 0.8)
    train = model_df.iloc[:split]
    val = model_df.iloc[split:]
    features = [c for c in model_df.columns if c not in ['observed_at', 'round_cutoff', 'target_up']]

    clf = HistGradientBoostingClassifier(
        max_iter=500,
        max_depth=4,
        learning_rate=0.03,
        min_samples_leaf=20,
        l2_regularization=0.5,
        early_stopping=True,
        random_state=42,
    )
    clf.fit(train[features], train['target_up'])
    proba = clf.predict_proba(val[features])[:, 1]
    pred = (proba >= 0.5).astype(int)

    metrics = {
        'rows': len(model_df),
        'train_rows': len(train),
        'val_rows': len(val),
        'accuracy': accuracy_score(val['target_up'], pred),
        'roc_auc': roc_auc_score(val['target_up'], proba),
        'brier': brier_score_loss(val['target_up'], proba),
        'log_loss': log_loss(val['target_up'], proba),
    }
    display(pd.Series(metrics).to_frame('value'))

## Next Questions To Answer

- Does accuracy improve or decay as seconds remaining gets lower?
- Are UP and DOWN symmetric, or is one side carrying the strategy?
- Are high edge signals actually better, or only more confident?
- Does Polymarket spread/liquidity filter bad entries?
- How many unique rounds do we have? Rows are repeated decision snapshots; rounds are the real independent examples.